# Equilibrium n-type and p-type doping

**Part IV skeleton — complete only after selecting and sourcing the Part III material**

This notebook extends the same equilibrium carrier model with literature-supported donor and acceptor charge states. It contains no Poisson equation, spatial band bending, current, mobility, or conductivity.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.constants import Boltzmann, electron_volt
from scipy.optimize import brentq

K_B_EV = Boltzmann / electron_volt


## 1. Dopant provenance and convention

For each species, record chemical identity, lattice site/defect type, energy level, charge states, degeneracy convention, concentration range, source, and compensation/amphoteric behavior.


In [2]:
dopant_provenance = pd.DataFrame([
    ("donor", "TODO", "TODO", "Ec - TODO", "TODO", "TODO", "TODO", "TODO", "TODO"),
    ("acceptor", "TODO", "TODO", "Ev + TODO", "TODO", "TODO", "TODO", "TODO", "TODO"),
], columns=["kind", "species", "site/defect", "level", "charge states",
            "degeneracy convention", "concentration range", "source",
            "compensation/amphoteric notes"])
dopant_provenance


,kind,species,site/defect,level,charge states,degeneracy convention,concentration range,source,compensation/amphoteric notes
0,donor,TODO,TODO,Ec - TODO,TODO,TODO,TODO,TODO,TODO
1,acceptor,TODO,TODO,Ev + TODO,TODO,TODO,TODO,TODO,TODO


## 2. Incomplete ionization

With the convention used in the assessment,

$$N_D^+=\frac{N_D}{1+g_D\exp[(\mu-E_D)/(k_BT)]},\qquad
N_A^-=\frac{N_A}{1+g_A\exp[(E_A-\mu)/(k_BT)]}.$$

Do not insert literature degeneracy factors until their convention has been reconciled with these exact equations. Test low- and high-temperature limits.


In [3]:
def ionized_donor_fraction(chemical_potential_ev, temperature_k,
                           donor_level_ev, donor_degeneracy):
    exponent = np.clip(
        (chemical_potential_ev - donor_level_ev) / (K_B_EV * temperature_k),
        -745.0, 709.0,
    )
    return 1.0 / (1.0 + donor_degeneracy * np.exp(exponent))


def ionized_acceptor_fraction(chemical_potential_ev, temperature_k,
                              acceptor_level_ev, acceptor_degeneracy):
    exponent = np.clip(
        (acceptor_level_ev - chemical_potential_ev) / (K_B_EV * temperature_k),
        -745.0, 709.0,
    )
    return 1.0 / (1.0 + acceptor_degeneracy * np.exp(exponent))

# Numerical smoke tests only; replace with material-specific limit tests.
assert 0.0 <= ionized_donor_fraction(0.0, 300.0, 0.1, 2.0) <= 1.0
assert 0.0 <= ionized_acceptor_fraction(0.0, 300.0, 0.1, 4.0) <= 1.0


## 3. Charge-neutrality solver

Solve

$$p+N_D^+-n-N_A^-=0$$

with a documented bracket and tolerances. Return $\mu,n,p,N_D^+,N_A^-$ and absolute/relative neutrality residuals in a `SemiconductorResult` object.


In [4]:
solver_result_schema = pd.DataFrame([
    ("temperature_K", "input"), ("mu_eV", "root"),
    ("n_cm3", "carrier result"), ("p_cm3", "carrier result"),
    ("ND_plus_cm3", "dopant result"), ("NA_minus_cm3", "dopant result"),
    ("neutrality_residual_cm3", "validation"),
    ("relative_residual", "validation"), ("bracket_eV", "validation"),
], columns=["field", "role"])
solver_result_schema


,field,role
0,temperature_K,input
1,mu_eV,root
2,n_cm3,carrier result
3,p_cm3,carrier result
4,ND_plus_cm3,dopant result
5,NA_minus_cm3,dopant result
6,neutrality_residual_cm3,validation
7,relative_residual,validation
8,bracket_eV,validation


## 4. Required equilibrium cases

Select concentrations and temperatures from material literature. The sweep should expose freeze-out, extrinsic behavior, intrinsic excitation, and—where physically justified—onset of degeneracy.


In [5]:
required_cases = pd.DataFrame([
    ("intrinsic", 0.0, 0.0, "TODO temperature range"),
    ("uncompensated n-type", np.nan, 0.0, "TODO"),
    ("compensated n-type", np.nan, np.nan, "TODO; ND > NA"),
    ("uncompensated p-type", 0.0, np.nan, "TODO"),
    ("compensated p-type", np.nan, np.nan, "TODO; NA > ND"),
], columns=["case", "ND (cm^-3)", "NA (cm^-3)", "range/constraint"])
required_cases


,case,ND (cm^-3),NA (cm^-3),range/constraint
0,intrinsic,0.0,0.0,TODO temperature range
1,uncompensated n-type,NaN,0.0,TODO
2,compensated n-type,NaN,NaN,TODO; ND > NA
3,uncompensated p-type,0.0,NaN,TODO
4,compensated p-type,NaN,NaN,TODO; NA > ND


## 5. Temperature sweeps

For every case, store $\mu(T)$, $n(T)$, $p(T)$, ionized concentrations/fractions, and neutrality residual. Identify regimes from numerical evidence rather than labels alone.


In [6]:
temperature_sweep_schema = pd.DataFrame(columns=[
    "case", "T (K)", "mu (eV)", "n (cm^-3)", "p (cm^-3)",
    "ND+ (cm^-3)", "NA- (cm^-3)", "donor ionization fraction",
    "acceptor ionization fraction", "relative neutrality residual",
])
temperature_sweep_schema


,case,T (K),mu (eV),n (cm^-3),p (cm^-3),ND+ (cm^-3),NA- (cm^-3),donor ionization fraction,acceptor ionization fraction,relative neutrality residual


## 6. Concentration sweeps

At one or more justified temperatures, calculate $n$ and $p$ versus nominal dopant concentration. Mark where nondegenerate statistics or the isolated-dopant model breaks down.


In [7]:
concentration_sweep_schema = pd.DataFrame(columns=[
    "kind", "T (K)", "nominal concentration (cm^-3)", "mu (eV)",
    "n (cm^-3)", "p (cm^-3)", "ionized fraction", "relative residual",
])
concentration_sweep_schema


,kind,T (K),nominal concentration (cm^-3),mu (eV),n (cm^-3),p (cm^-3),ionized fraction,relative residual


## 7. Required plots

- $\mu(T)$ relative to $E_V,E_C,E_D,E_A$
- log-scale $n(T)$ and $p(T)$
- donor and acceptor ionization fractions
- carriers versus nominal concentration
- neutrality residual over every sweep
- representative equilibrium energy-level diagrams

Energy-level diagrams must not suggest spatial band bending.


In [8]:
plot_checklist = pd.DataFrame([
    ("mu versus T with all levels", "TODO"),
    ("n and p versus T", "TODO"),
    ("ionization fractions", "TODO"),
    ("carriers versus concentration", "TODO"),
    ("neutrality residuals", "TODO"),
    ("intrinsic/n-type/p-type energy-level diagrams", "TODO"),
], columns=["plot", "status"])
plot_checklist


,plot,status
0,mu versus T with all levels,TODO
1,n and p versus T,TODO
2,ionization fractions,TODO
3,carriers versus concentration,TODO
4,neutrality residuals,TODO
5,intrinsic/n-type/p-type energy-level diagrams,TODO


## 8. Approximation comparisons

Compare incomplete ionization against full ionization and identify where majority-carrier error is below 5%. For narrow-gap or degenerate cases, also compare Fermi–Dirac and Boltzmann statistics.


In [9]:
approximation_comparison_schema = pd.DataFrame(columns=[
    "case", "T (K)", "dopant concentration (cm^-3)",
    "majority carrier incomplete (cm^-3)", "majority carrier full (cm^-3)",
    "full-ionization relative error", "FD/Boltzmann relative difference",
])
approximation_comparison_schema


,case,T (K),dopant concentration (cm^-3),majority carrier incomplete (cm^-3),majority carrier full (cm^-3),full-ionization relative error,FD/Boltzmann relative difference


## 9. Scientific conclusion — TODO

Answer quantitatively: over what temperature and concentration range is the effective-mass plus equilibrium dopant model adequate, and which omitted physics invalidates it first? Cite both calculations and literature evidence.
